# MavunoWise V3 — Phase 1B Data Pipeline**Strict Rules:**1. NO forward-filling/interpolation of TARGET prices (prevents look-ahead leakage).2. Canonical unit is KES/kg.3. Internal data must pass PII stripping and ML consent gate.4. Raw data is immutable.5. Time-series splits must be chronological.

## Step 1: Setup & Imports*Dependencies: pandas, numpy, pyarrow, pyyaml*

In [ ]:
import pandas as pdimport numpy as npimport osimport hashlibimport yamlfrom datetime import datetime# Load configswith open('../configs/source_registry.yaml', 'r') as f:    sources = yaml.safe_load(f)with open('../configs/imputation_policy.yaml', 'r') as f:    imputation_rules = yaml.safe_load(f)

## Step 2: Raw Data Ingestion & SnapshotsLoad raw CSVs. Do not mutate.

In [ ]:
# df_kamis = pd.read_csv('../data/raw/kamis_export.csv')# df_internal_tx = pd.read_csv('../data/raw/internal_transactions.csv')

## Step 3: PII / Consent Gate (Internal Data)Strip PII. Apply deterministic pseudonymization. Do not use row-order.

In [ ]:
def pseudonymize_id(raw_id, salt="CHANGE_ME_ENV_VAR"):    return "FARMER_" + hashlib.sha256((str(raw_id) + salt).encode()).hexdigest()[:12]# df_internal_tx['farmer_pseudo_id'] = df_internal_tx['farmer_id'].apply(pseudonymize_id)# df_internal_tx = df_internal_tx.drop(columns=['farmer_id', 'phone', 'name'])

## Step 4: Unit NormalizationConvert everything to KES/kg.

In [ ]:
BAG_SIZE_KG = 90def to_kes_per_kg(row):    # Example: if price is per 90kg bag    return row['price_per_bag'] / BAG_SIZE_KG

## Step 5: Geography & Commodity NormalizationMap source names to canonical MavunoWise names.

In [ ]:
# mapping_dict = {#     "Kitale Market": "Kitale",#     "maize grain": "MAIZE"# }

## Step 6: Point-in-Time As-Of Join**CRITICAL:** No forward-filling of market prices.For a transaction at time T, only use source info published on or before T.

In [ ]:
# df_joined = pd.merge_asof(#     df_transactions.sort_values('timestamp'),#     df_market_prices.sort_values('observed_at'),#     left_on='timestamp',#     right_on='observed_at',#     direction='backward', # Only look backward in time#     by=['county', 'commodity']# )

## Step 7: ML Target Generation (Horizon Shifting)Generate 7, 14, and 30-day future price targets. *Note: This is the ONLY time we look forward, and it is strictly to create the y-variable.*

In [ ]:
# df_joined = df_joined.sort_values(['county', 'market', 'commodity', 'timestamp'])# df_joined['target_price_7d'] = df_joined.groupby(['county', 'market', 'commodity'])['market_price_kes_kg'].shift(-7)# df_joined['target_price_14d'] = df_joined.groupby(['county', 'market', 'commodity'])['market_price_kes_kg'].shift(-14)# df_joined['target_price_30d'] = df_joined.groupby(['county', 'market', 'commodity'])['market_price_kes_kg'].shift(-30)

## Step 8: Time-Series Feature Scaffolding (Lags & Rolling)Create historical features for the ML model. *Must be calculated per group to prevent leakage across time boundaries.*

In [ ]:
# # Lag features# df_joined['price_lag_7d'] = df_joined.groupby(['county', 'commodity'])['market_price_kes_kg'].shift(7)# # # Rolling window features (must be closed on the left to prevent leakage)# df_joined['rolling_mean_price_7d'] = df_joined.groupby(['county', 'commodity'])['market_price_kes_kg'].transform(#     lambda x: x.shift(1).rolling(7, closed='left').mean()# )

## Step 9: Data Quality, Outliers & ImputationApply rules from imputation_policy.yaml. Flag, do not blindly delete.

In [ ]:
# # Example: Drop rows where target is missing# df_clean = df_joined.dropna(subset=['target_price_7d', 'target_price_14d', 'target_price_30d'])

## Step 10: Chronological Train/Test Split MarkingCreate temporal splits for walk-forward evaluation in Phase 2B.

In [ ]:
# # Define cutoff dates# train_cutoff = pd.to_datetime('2023-06-01')# val_cutoff = pd.to_datetime('2023-09-01')# # def assign_split(date):#     if date <= train_cutoff: return 'TRAIN'#     elif date <= val_cutoff: return 'VALIDATION'#     else: return 'TEST'# # df_clean['dataset_split'] = df_clean['timestamp'].apply(assign_split)

## Step 11: Leakage Audit & Coverage ReportEnsure no post-sale info, settlement info, or actual yield is used as a feature.Determine if we have enough data.

In [ ]:
# forbidden_features = ['settlement_date', 'actual_yield_bags', 'buyer_phone']# # total_rows = len(df_clean)# if total_rows < 1000:#     print("WARNING: INSUFFICIENT DATA FOR MODEL TRAINING")

## Step 12: Versioned Export & ManifestExport to Parquet. Generate checksum.

In [ ]:
# version = "2026.10.07.v001"# filename = f"../exports/mavunowise_market_dataset_{version}.parquet"# df_clean.to_parquet(filename)